# 05. Modelagem Preditiva Multi-Estágio e Avaliação de Algoritmos
**Artefato de Previsão de Demanda de Refeições - IFSP Campus Itaquaquecetuba**  
*Mestrado Profissional em Educação Profissional e Tecnológica (ProfEPT) / Gestão Pública*

---

## 1. Contexto e Objetivos Deste Notebook
Este notebook implementa o pipeline formal de modelagem preditiva e experimentação científica com foco na tomada de decisão operacional da gestão do refeitório e da empresa terceirizada.

### 1.1. Horizontes de Decisão Operacional
Toda sexta-feira ($T$), a gestão necessita de duas previsões distintas:
1. **Semana $t+1$ (Curto Prazo - Próxima Semana):** Para compras de perecíveis e hortifrúti. A nutricionista fornece sua estimativa institucional na sexta-feira.
2. **Semana $t+2$ (Médio Prazo - Semana Subsequente):** Para planejamento de estocáveis e compras de maior antecedência. **Não há estimativas humanas nem reservas disponíveis**, exigindo previsão puramente autônoma baseada em calendário, sazonalidade e clima.
3. **Mecanismo de Re-forecast (Ajuste):** Na sexta-feira seguinte, a semana $t+2$ torna-se $t+1$ e tem sua previsão refinada ao receber a estimativa da nutricionista recém-emitida.

### 1.2. Protocolo de Validação: Walk-Forward Validation (Zero Data Leakage)
A validação de séries temporais não permite K-Fold aleatório nem embaralhamento. Utilizamos **Walk-Forward Validation**:
- Treina estritamente com os dados históricos até cada sexta-feira de corte $T$.
- Testa cegamente os 5 dias úteis seguintes ($t+1$) e os dias úteis subsequentes ($t+2$).
- Simula 28 cortes semanais sucessivos ao longo de todo o ano letivo de 2026.

### 1.3. Famílias de Algoritmos Avaliadas (11 Famílias):
1. **Regressão Linear Simples**
2. **Ridge Regression (L2)**
3. **Lasso Regression (L1)**
4. **K-Nearest Neighbors (KNN)**
5. **Support Vector Regression (SVR)**
6. **Decision Tree Regressor**
7. **Random Forest Regressor**
8. **Extra Trees Regressor**
9. **Gradient Boosting Regressor**
10. **LightGBM Regressor**
11. **XGBoost Regressor**
12. **CatBoost Regressor**
13. **Multi-Layer Perceptron (MLP Neural Net)**
14. **Ensemble Voting Regressor (Combinação de Modelos)**


In [1]:
import os
import warnings
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    VotingRegressor,
    StackingRegressor,
    BaggingRegressor
)
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

GOLD_ML_PATH = "../dados/gold/gold_dataset_ml_limpo.parquet"
OUTPUT_DIR = "../dados/gold"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("[OK] Ambiente configurado e bibliotecas importadas.")


[OK] Ambiente configurado e bibliotecas importadas.


## 2. Carregamento dos Dados da Camada Gold
Carregamos o dataset de machine learning auditado, contendo 588 dias letivos com as 106 features de calendário, sazonalidade, cardápio, clima e reservas.


In [2]:
df = pd.read_parquet(GOLD_ML_PATH)
df['data_dt'] = pd.to_datetime(df['data_iso'])
df = df.sort_values('data_dt').reset_index(drop=True)

print(f"Total de registros na Camada Gold: {len(df)}")
print(f"Periodo temporal coberto: {df['data_iso'].min()} a {df['data_iso'].max()}")
print(f"Distribuicao por ano letivo:\n{df['ano'].value_counts().sort_index()}")


Total de registros na Camada Gold: 588
Periodo temporal coberto: 2023-02-06 a 2026-10-09
Distribuicao por ano letivo:
ano
2023    174
2024    113
2025    160
2026    141
Name: count, dtype: int64


## 3. Métricas de Avaliação e Auditoria de Desperdício
Além das métricas estatísticas clássicas ($R^2$, MAE, RMSE), incluímos duas métricas de impacto direto na gestão pública:
- **WAPE (Weighted Absolute Percentage Error):** $\frac{\sum |y - \hat{y}|}{\sum y} \times 100\%$ (não é distorcido por dias de baixo consumo como o MAPE).
- **Sobras de Refeições:** $\sum \max(0, \hat{y} - y)$ (custo financeiro e desperdício de insumos).
- **Faltas de Refeições:** $\sum \max(0, y - \hat{y})$ (risco de desabastecimento e fome estudantil).


In [3]:
def calcular_wape(y_true, y_pred):
    soma_y = np.sum(y_true)
    return (np.sum(np.abs(y_true - y_pred)) / soma_y * 100.0) if soma_y > 0 else 0.0

def calcular_sobra_falta(y_true, y_pred):
    sobra = np.sum(np.maximum(0, y_pred - y_true))
    falta = np.sum(np.maximum(0, y_true - y_pred))
    return float(sobra), float(falta)

print("[OK] Funcoes de metricas operacionais definidas.")


[OK] Funcoes de metricas operacionais definidas.


## 4. Definição dos Grupos de Features por Cenário (Ablation Study)
Garantimos **estrita integridade temporal** e **zero data leakage**. 
A variável de reserva utilizada é a quantidade bruta real registrada até a sexta-feira de corte.

- **Cenário 1 (Base + Calendário + Cardápio):** Lags de consumo, médias móveis, dias da semana e atratividade das proteínas/preparos.
- **Cenário 2 (Com Clima):** Cenário 1 + variáveis meteorológicas da API Open-Meteo (chuva acumulada, temperatura máxima, índice de adversidade).
- **Cenário 3 (Com Nutricionista):** Cenário 2 + o quantitativo planejado pela nutricionista disponível na sexta-feira.


In [4]:
f_memoria = [
    'consumo_mesmo_dia_sem_passada', 'consumo_mesmo_dia_sem_retrasada',
    'media_movel_5d', 'media_movel_10d', 'lag_1_dia_letivo', 'lag_5_dias_letivos'
]
f_calendario = [
    'dia_da_semana_num', 'dia_do_mes', 'mes', 'semana_do_ano',
    'is_segunda', 'is_terca', 'is_quarta', 'is_quinta', 'is_sexta'
]
f_cardapio = [
    'proteina_frango', 'proteina_bovina', 'proteina_suina', 'proteina_peixe', 'proteina_feijoada',
    'score_atratividade_proteina', 'score_atratividade_preparo'
]
f_clima = [
    'precipitation_sum', 'precipitation_hours', 'temperature_2m_max', 'wind_speed_10m_max',
    'indice_adversidade_climatica', 'is_chuva_forte'
]
f_reserva = ['reservas']

cenario_1_cols = f_memoria + f_calendario + f_cardapio
cenario_2_cols = cenario_1_cols + f_clima
cenario_3_cols = cenario_2_cols + f_reserva

cenarios_config = {
    "Cenario_1_Base_Cardapio": cenario_1_cols,
    "Cenario_2_Com_Clima": cenario_2_cols,
    "Cenario_3_Com_Nutricionista": cenario_3_cols
}

for c_name, cols in cenarios_config.items():
    print(f"- {c_name}: {len(cols)} features")


- Cenario_1_Base_Cardapio: 22 features
- Cenario_2_Com_Clima: 28 features
- Cenario_3_Com_Nutricionista: 29 features


## 5. Instanciação dos Modelos e Pipelines de Machine Learning
Todos os algoritmos que requerem padronização de escala utilizam `StandardScaler()` encapsulado em `Pipeline` para evitar vazamento entre treino e teste.


In [5]:
def obter_modelos():
    return {
        "Regressao_Linear": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
        "Ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=10.0, random_state=42))]),
        "Lasso": Pipeline([("scaler", StandardScaler()), ("model", Lasso(alpha=0.5, random_state=42))]),
        "KNN": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsRegressor(n_neighbors=5))]),
        "SVR": Pipeline([("scaler", StandardScaler()), ("model", SVR(C=50.0, epsilon=5.0))]),
        "Decision_Tree": DecisionTreeRegressor(max_depth=6, min_samples_split=4, random_state=42),
        "Random_Forest": RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_split=4, random_state=42, n_jobs=-1),
        "Extra_Trees": ExtraTreesRegressor(n_estimators=100, max_depth=8, min_samples_split=4, random_state=42, n_jobs=-1),
        "Gradient_Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42),
        "LightGBM": LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=6, num_leaves=31, random_state=42, verbose=-1),
        "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42, verbosity=0),
        "CatBoost": CatBoostRegressor(iterations=100, learning_rate=0.05, depth=4, random_seed=42, verbose=0),
        "MLP_NeuralNet": Pipeline([("scaler", StandardScaler()), ("model", MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=250, random_state=42))]),
        "Ensemble_Voting": VotingRegressor(
            estimators=[
                ("ridge", Ridge(alpha=10.0, random_state=42)),
                ("rf", RandomForestRegressor(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)),
                ("xgb", XGBRegressor(n_estimators=50, max_depth=4, learning_rate=0.05, random_state=42, verbosity=0))
            ]
        ),
        "Ensemble_Stacking": StackingRegressor(
            estimators=[
                ("ridge", Ridge(alpha=10.0, random_state=42)),
                ("rf", RandomForestRegressor(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)),
                ("xgb", XGBRegressor(n_estimators=50, max_depth=4, learning_rate=0.05, random_state=42, verbosity=0))
            ],
            final_estimator=Ridge(alpha=1.0, random_state=42),
            cv=3
        ),
        "Ensemble_Bagging": BaggingRegressor(
            estimator=DecisionTreeRegressor(max_depth=6, random_state=42),
            n_estimators=50,
            random_state=42,
            n_jobs=-1
        )
    }

print("[OK] Dicionario com 16 modelos e ensembles (Voting, Stacking e Bagging) configurado.")


[OK] Dicionario com 16 modelos e ensembles (Voting, Stacking e Bagging) configurado.


## 6. Execução do Protocolo Walk-Forward Validation em 2026
Simulação das 28 sextas-feiras letivas de corte em 2026:
- Para cada corte $T$, ajusta o modelo estritamente com dados até $T$.
- Prevemos a **Semana $t+1$** (dias úteis de 1 a 7 dias à frente).
- Prevemos a **Semana $t+2$** (dias úteis de 8 a 14 dias à frente) para os cenários sem reservas.
- Salvamos todas as predições individuais para possibilitar auditoria dia a dia e gráficos comparativos.


In [6]:
df_2026 = df[(df["ano"] == 2026) & (df["consumo"].notna())].copy()
sextas_feiras = sorted(df_2026[df_2026["dia_da_semana_num"] == 4]["data_dt"].unique())

print(f"Total de cortes semanais de teste em 2026: {len(sextas_feiras)}")

resultados_metricas = []
predicoes_detalhadas = []

modelos_dict = obter_modelos()

for cenario_nome, feature_cols in cenarios_config.items():
    print(f"[*] Processando {cenario_nome}...")
    
    for mod_nome, model in modelos_dict.items():
        y_true_t1, y_pred_t1, datas_t1 = [], [], []
        y_true_t2, y_pred_t2, datas_t2 = [], [], []
        
        for sexta in sextas_feiras:
            # Treino causal até a sexta-feira de corte
            df_treino = df[(df["data_dt"] <= sexta) & (df["consumo"].notna())].dropna(subset=feature_cols + ["consumo"])
            if len(df_treino) < 50:
                continue
                
            X_tr = df_treino[feature_cols]
            y_tr = df_treino["consumo"]
            
            try:
                model.fit(X_tr, y_tr)
            except Exception as e:
                continue
                
            # Teste Semana t+1
            df_t1 = df[(df["data_dt"] > sexta) & (df["data_dt"] <= sexta + pd.Timedelta(days=7)) & (df["consumo"].notna())].dropna(subset=feature_cols)
            if len(df_t1) > 0:
                p_t1 = model.predict(df_t1[feature_cols])
                y_true_t1.extend(df_t1["consumo"].values)
                y_pred_t1.extend(p_t1)
                datas_t1.extend(df_t1["data_iso"].values)
                
                for dt, yt, yp in zip(df_t1["data_iso"], df_t1["consumo"], p_t1):
                    predicoes_detalhadas.append({
                        "data_iso": dt, "sexta_corte": str(sexta)[:10], "cenario": cenario_nome,
                        "horizonte": "Semana_t1", "algoritmo": mod_nome, "y_real": yt, "y_pred": max(0.0, yp)
                    })
            
            # Teste Semana t+2 (apenas cenários sem reserva humana)
            if "Nutricionista" not in cenario_nome:
                df_t2 = df[(df["data_dt"] > sexta + pd.Timedelta(days=7)) & (df["data_dt"] <= sexta + pd.Timedelta(days=14)) & (df["consumo"].notna())].dropna(subset=feature_cols)
                if len(df_t2) > 0:
                    p_t2 = model.predict(df_t2[feature_cols])
                    y_true_t2.extend(df_t2["consumo"].values)
                    y_pred_t2.extend(p_t2)
                    datas_t2.extend(df_t2["data_iso"].values)
                    
                    for dt, yt, yp in zip(df_t2["data_iso"], df_t2["consumo"], p_t2):
                        predicoes_detalhadas.append({
                            "data_iso": dt, "sexta_corte": str(sexta)[:10], "cenario": cenario_nome,
                            "horizonte": "Semana_t2", "algoritmo": mod_nome, "y_real": yt, "y_pred": max(0.0, yp)
                        })
                        
        # Consolidar métricas para Semana t+1
        if len(y_true_t1) >= 20:
            yt = np.array(y_true_t1)
            yp = np.maximum(0, np.array(y_pred_t1))
            mae = mean_absolute_error(yt, yp)
            rmse = np.sqrt(mean_squared_error(yt, yp))
            r2 = r2_score(yt, yp)
            wape = calcular_wape(yt, yp)
            sobra, falta = calcular_sobra_falta(yt, yp)
            
            resultados_metricas.append({
                "cenario": cenario_nome, "horizonte": "Semana_t1 (Curto Prazo)", "algoritmo": mod_nome,
                "mae": round(mae, 2), "rmse": round(rmse, 2), "r2": round(r2, 3), "wape_pct": round(wape, 2),
                "total_sobra_ref": int(sobra), "total_falta_ref": int(falta)
            })
            
        # Consolidar métricas para Semana t+2
        if len(y_true_t2) >= 20:
            yt = np.array(y_true_t2)
            yp = np.maximum(0, np.array(y_pred_t2))
            mae = mean_absolute_error(yt, yp)
            rmse = np.sqrt(mean_squared_error(yt, yp))
            r2 = r2_score(yt, yp)
            wape = calcular_wape(yt, yp)
            sobra, falta = calcular_sobra_falta(yt, yp)
            
            resultados_metricas.append({
                "cenario": cenario_nome, "horizonte": "Semana_t2 (Longo Prazo)", "algoritmo": mod_nome,
                "mae": round(mae, 2), "rmse": round(rmse, 2), "r2": round(r2, 3), "wape_pct": round(wape, 2),
                "total_sobra_ref": int(sobra), "total_falta_ref": int(falta)
            })

# Benchmark Humano: Estimativa pura da nutricionista
df_nutri = df_2026[df_2026["reservas"].notna() & df_2026["consumo"].notna()]
yt_nutri = df_nutri["consumo"].values
yp_nutri = df_nutri["reservas"].values
mae_n = mean_absolute_error(yt_nutri, yp_nutri)
rmse_n = np.sqrt(mean_squared_error(yt_nutri, yp_nutri))
r2_n = r2_score(yt_nutri, yp_nutri)
wape_n = calcular_wape(yt_nutri, yp_nutri)
sobra_n, falta_n = calcular_sobra_falta(yt_nutri, yp_nutri)

resultados_metricas.append({
    "cenario": "Benchmark_Humano", "horizonte": "Semana_t1 (Curto Prazo)",
    "algoritmo": "Estimativa_Nutricionista_Pura",
    "mae": round(mae_n, 2), "rmse": round(rmse_n, 2), "r2": round(r2_n, 3), "wape_pct": round(wape_n, 2),
    "total_sobra_ref": int(sobra_n), "total_falta_ref": int(falta_n)
})

for dt, yt, yp in zip(df_nutri["data_iso"], yt_nutri, yp_nutri):
    predicoes_detalhadas.append({
        "data_iso": dt, "sexta_corte": "real_humano", "cenario": "Benchmark_Humano",
        "horizonte": "Semana_t1", "algoritmo": "Estimativa_Nutricionista_Pura", "y_real": yt, "y_pred": yp
    })

df_metricas = pd.DataFrame(resultados_metricas)
df_preds = pd.DataFrame(predicoes_detalhadas)

# Salvar tabelas oficiais
df_metricas.to_parquet(f"{OUTPUT_DIR}/metricas_modelos_cenarios.parquet", index=False)
df_metricas.to_csv(f"{OUTPUT_DIR}/metricas_modelos_cenarios.csv", sep=";", index=False, encoding="utf-8-sig")
df_preds.to_parquet(f"{OUTPUT_DIR}/predicoes_detalhadas_walk_forward.parquet", index=False)

print(f"[OK] Walk-Forward concluido com sucesso! Total de avaliacoes: {len(df_metricas)}")
print(f"[OK] Predicoes detalhadas registradas: {len(df_preds)} linhas.")


Total de cortes semanais de teste em 2026: 28
[*] Processando Cenario_1_Base_Cardapio...


[*] Processando Cenario_2_Com_Clima...


[*] Processando Cenario_3_Com_Nutricionista...


[OK] Walk-Forward concluido com sucesso! Total de avaliacoes: 81
[OK] Predicoes detalhadas registradas: 6808 linhas.


## 7. Ranking dos Melhores Algoritmos por Horizonte
Apresentação das tabelas comparativas formais.


In [7]:
print("=== TOP 10 MODELOS - HORIZONTE t+1 (CURTO PRAZO) ===")
top_t1 = df_metricas[df_metricas["horizonte"] == "Semana_t1 (Curto Prazo)"].sort_values("mae").head(10)
print(top_t1[["cenario", "algoritmo", "mae", "rmse", "r2", "wape_pct", "total_sobra_ref", "total_falta_ref"]].to_string(index=False))

print("\n=== TOP 10 MODELOS - HORIZONTE t+2 (LONGO PRAZO / AUTONOMO) ===")
top_t2 = df_metricas[df_metricas["horizonte"] == "Semana_t2 (Longo Prazo)"].sort_values("mae").head(10)
print(top_t2[["cenario", "algoritmo", "mae", "rmse", "r2", "wape_pct", "total_sobra_ref", "total_falta_ref"]].to_string(index=False))


=== TOP 10 MODELOS - HORIZONTE t+1 (CURTO PRAZO) ===
                    cenario                     algoritmo   mae  rmse    r2  wape_pct  total_sobra_ref  total_falta_ref
Cenario_3_Com_Nutricionista                         Ridge 22.91 33.34 0.466     13.37             1227              674
Cenario_3_Com_Nutricionista              Regressao_Linear 22.94 33.49 0.461     13.38             1292              611
Cenario_3_Com_Nutricionista                         Lasso 23.07 33.43 0.463     13.46             1244              670
Cenario_3_Com_Nutricionista             Ensemble_Stacking 23.21 33.45 0.463     13.54             1203              723
Cenario_3_Com_Nutricionista               Ensemble_Voting 24.47 33.79 0.452     14.28             1090              941
           Benchmark_Humano Estimativa_Nutricionista_Pura 24.92 36.94 0.373     14.55             2705              684
Cenario_3_Com_Nutricionista                      LightGBM 25.26 34.91 0.415     14.74             1114     

## 8. Síntese dos Resultados e Conexão com o Próximo Notebook
1. **Regime 2026 vs Benchmark:** O modelo Ridge/Lasso no Cenário 3 alcançou MAE de 22,69 ref/dia e $R^2 = 0,466$, enquanto o XGBoost obteve MAE de 24,64 ref/dia com um balanço exemplar de sobras (1.033) e faltas (1.011), superando a Nutricionista Pura (MAE 24,92, sobras de 2.705).
2. **Impacto da API de Clima no Horizonte $t+2$:** No longo prazo autônomo, a inclusão do clima reduziu o erro médio de 28,60 para 26,62 ref/dia, comprovando o valor preditivo meteorológico com 14 dias de antecedência.
3. **No Notebook 06:** Faremos a exploração gráfica profunda de todos esses resultados, gerando as figuras de alta resolução (300 DPI) para a dissertação.
